# SkyQuery E3b: same as E3, with the text-alignment layers frozen

E3 taught YOLOE 35 drone concepts (tree 1 → 52 AP) but it stopped answering to every word it wasn't taught. The
diagnosis (`reports/ovd_e3_wise.json`): the contrastive head's BatchNorm statistics and the text adapter (RepRTA)
drifted. E3b changes **only that**: those layers stay at their zero-shot values. Same SAM 3 labels (reused from E3,
no relabelling), same vocabulary, same held-out words, same training time (6.9 h), so any difference is the freeze.

**Before you start:** upload your `e3_results.zip` as a Kaggle dataset named `skyquery-e3-labels` and add it; add the
new `skyquery-code` (it includes the E3 weights for the comparison); GPU T4 ×2; Internet on. No HF token needed.
Run once with `QUICK = True`, then Save & Run All (Commit). About 7.5 hours.

Pre-registered in the team's pre-registration notes, round 5.

In [ ]:
# --- Settings ---------------------------------------------------------------------
QUICK = False          # True = 20-minute dry run on a few frames. Do this once interactively before the overnight run.
MODEL_SIZE = "s"       # "s": fast, what the app uses. "m": better but ~2x slower to train and to run.
TEACHER = "sam3"       # only used if LABEL_HOURS > 0 (labels are reused from E3)
TOTAL_HOURS = 9.0      # Kaggle stops a session at 12 h, keep a margin
LABEL_HOURS = 0.0      # 0 = reuse E3's SAM 3 labels exactly (keeps the comparison clean)
TRAIN_HOURS = 6.9      # same as E3
EVAL_HOURS = 0.6
TAU = 0.5              # teacher score needed for a box to become a training label

import time
T0 = time.time()
def hours_left():
    return TOTAL_HOURS - (time.time() - T0) / 3600

# --- Code, packages, secrets ----------------------------------------------------------
# Needs: the skyquery-code dataset added, GPU T4 x2, Internet on, and an HF_TOKEN secret (read-only token).
import os, subprocess, sys, glob, shutil, json
CODE = glob.glob("/kaggle/input/skyquery-code/**/pyproject.toml", recursive=True)
assert CODE, "Add the skyquery-code dataset to this notebook"
WORK = "/kaggle/working/skyquery"
if not os.path.exists(WORK):
    shutil.copytree(os.path.dirname(CODE[0]), WORK, ignore=shutil.ignore_patterns("node_modules", "data", "cache", "frontend"))
os.chdir(WORK)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[eval]", "transformers==5.20.2", "ultralytics==8.4.182"], check=True)
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded from Kaggle secrets")
except Exception:
    print("No HF_TOKEN secret: SAM 3 is gated, so the teacher will fall back to OWLv2")

SCRATCH = "/tmp/e3"                      # big, temporary files (not saved as notebook output)
os.makedirs(f"{SCRATCH}/data", exist_ok=True); os.makedirs("cache", exist_ok=True); os.makedirs("reports", exist_ok=True)
BASE = "https://github.com/ultralytics/assets/releases/download/v0.0.0/"
def fetch(split):
    d = f"{SCRATCH}/data/VisDrone2019-DET-{split}"
    if not os.path.exists(d):
        subprocess.run(f"curl -sSL -o /tmp/{split}.zip {BASE}VisDrone2019-DET-{split}.zip && unzip -q /tmp/{split}.zip -d {SCRATCH}/data && rm /tmp/{split}.zip", shell=True, check=True)
    return d
TRAIN, VAL = fetch("train"), fetch("val")
print(len(os.listdir(f"{TRAIN}/images")), "train frames,", len(os.listdir(f"{VAL}/images")), "val frames")


In [ ]:
# --- Reuse E3's SAM 3 labels; YOLOE weights and text encoder ---------------------------
LAB = glob.glob("/kaggle/input/skyquery-e3-labels/**/teacher_val_0.npz", recursive=True)
assert LAB, "Add your e3_results.zip as a dataset named skyquery-e3-labels"
for f in glob.glob(os.path.join(os.path.dirname(LAB[0]), "teacher_*.npz")):
    shutil.copy(f, "cache/")
print(sorted(os.listdir("cache")))
from ultralytics import YOLOE
_m = YOLOE(f"yoloe-26{MODEL_SIZE}-seg.pt"); _m.get_text_pe(["tree"]); del _m
assert os.path.exists(f"weights/yoloe-26{MODEL_SIZE}-drone-ov.pt"), "the E3 weights should be inside skyquery-code/weights"


In [ ]:
# --- Teacher labels: all val frames first, then train frames until the deadline ---------
def run_on_gpus(jobs, deadline):
    procs = []
    for gpu, (args, log) in enumerate(jobs):
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
        cmd = [sys.executable, "-m", "skyquery.ovtrain.label", *args, "--device", "cuda:0", "--teacher", TEACHER,
               "--deadline", str(deadline), "--shard", str(gpu), "--nshards", str(len(jobs))]
        procs.append((subprocess.Popen(cmd, env=env, stdout=open(log, "w"), stderr=subprocess.STDOUT), log))
    while any(p.poll() is None for p, _ in procs):
        time.sleep(300 if not QUICK else 20)
        for _, log in procs:
            lines = open(log).read().strip().splitlines()
            print(lines[-1] if lines else f"{log}: starting")
    for p, log in procs:
        print(open(log).read()[-1500:])
        if p.returncode:
            raise RuntimeError(f"labelling failed, see {log}")

LIM = ["--limit", "16"] if QUICK else []
label_end = time.time() + (0.25 if QUICK else LABEL_HOURS) * 3600
if LABEL_HOURS > 0:
    run_on_gpus([(["--split", VAL, "--group", "all", "--out", f"cache/teacher_val_{g}.npz", *LIM], f"/tmp/label_val_{g}.log") for g in (0, 1)], label_end)
    run_on_gpus([(["--split", TRAIN, "--group", "seen", "--out", f"cache/teacher_train_{g}.npz", *LIM], f"/tmp/label_train_{g}.log") for g in (0, 1)], label_end)
print("labelling:", "reused from E3" if LABEL_HOURS <= 0 else "done", "| hours left:", round(hours_left(), 2))


In [ ]:
# --- Build the training sets ----------------------------------------------------------
DS = f"{SCRATCH}/ds"
cmd = [sys.executable, "-m", "skyquery.ovtrain.build", "--train", TRAIN, "--val", VAL, "--out", DS, "--tau", str(TAU), "--workers", "4",
       "--teacher-train", "cache/teacher_train_*.npz", "--teacher-val", "cache/teacher_val_*.npz"] + (["--limit", "16"] if QUICK else [])
subprocess.run(cmd, check=True)
stats = json.load(open(f"{DS}/stats.json"))
print({k: v for k, v in stats.items() if k != "train_boxes_per_class"})
print("teacher boxes kept (train):", {k: v for k, v in stats["train_boxes_per_class"].items() if v})


In [ ]:
# --- Train (time-boxed; falls back to one GPU if two-GPU training fails) ----------------
OUT_W = f"/kaggle/working/yoloe-26{MODEL_SIZE}-drone-ov-b.pt"
train_hours = 0 if QUICK else min(TRAIN_HOURS, max(0.5, hours_left() - EVAL_HOURS - 0.25))
print(f"training for {train_hours:.2f} h")
cmd = [sys.executable, "-m", "skyquery.ovtrain.train", "--data", f"{DS}/data.json", "--size", MODEL_SIZE, "--hours", str(train_hours),
       "--device", "0" if QUICK else "0,1", "--project", "/kaggle/working/runs", "--name", "e3b_ov", "--freeze-align", "--out", OUT_W, "--workers", "4"]
if QUICK:
    cmd += ["--epochs", "1"]
subprocess.run(cmd, check=True)
print("hours left:", round(hours_left(), 2))


In [ ]:
# --- Score zero-shot vs tuned on val, write the app config -----------------------------
name = f"weights/yoloe-26{MODEL_SIZE}-drone-ov-b.pt"
cmd = [sys.executable, "-m", "skyquery.ovtrain.evaluate", "--val", VAL, "--teacher-val", "cache/teacher_val_0.npz", "cache/teacher_val_1.npz",
       "--model", f"zeroshot=yoloe-26{MODEL_SIZE}-seg.pt", "--model", f"tuned={OUT_W}", "--model", f"e3=weights/yoloe-26{MODEL_SIZE}-drone-ov.pt", "--device", "0", "--tau", str(TAU),
       "--out", "reports/ovd_e3b_val.json", "--write-config", "configs/ovd_drone.json", "--weights-name", name]
if QUICK:
    cmd += ["--limit", "16", "--boot", "20"]
subprocess.run(cmd, check=True)


In [ ]:
# --- Package: download these from the Output tab --------------------------------------
shutil.copy("reports/ovd_e3b_val.json", "/kaggle/working/ovd_e3b_val.json")
shutil.copy("configs/ovd_drone.json", "/kaggle/working/ovd_drone.json")
shutil.copy(f"{DS}/stats.json", "reports/e3b_dataset_stats.json")
subprocess.run("zip -qr /kaggle/working/e3b_results.zip reports/ovd_e3b_val.json reports/e3b_dataset_stats.json configs/ovd_drone.json", shell=True)
for f in ("e3b_results.zip", os.path.basename(OUT_W)):
    print(f, round(os.path.getsize(f"/kaggle/working/{f}") / 1e6, 1), "MB")
print("Put", os.path.basename(OUT_W), "in skyquery/weights/ and ovd_drone.json in skyquery/configs/; the app picks them up on restart.")
print("total hours:", round((time.time() - T0) / 3600, 2))
